# 01 · Authentication walkthrough

This notebook explains how the tool authenticates to a Microsoft 365 tenant.
It is **read-only** and, for reproducibility, does **not** perform a live sign-in
(that requires a real tenant admin and a browser). Instead it shows the exact
code paths the web app and CLI use.

> The tool never stores tokens on disk and requests only read scopes.

In [ ]:
import sys, json
from pathlib import Path

# Repo root = parent of the notebooks/ directory.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
FIXTURES = REPO / "tests" / "fixtures"
print("repo:", REPO)
print("fixtures:", [p.name for p in FIXTURES.glob("*.json")])

## The public client

One multi-tenant app registration (no client secret) is reused across all
client tenants. Authority is `/organizations` so any work/school tenant can
sign in; the audited tenant is identified afterwards from the id-token `tid`
claim.

In [ ]:
from m365_review.settings import Settings

# A throwaway Settings with a placeholder client id (no real sign-in here).
settings = Settings(AZURE_APP_CLIENT_ID="00000000-0000-0000-0000-000000000000")
print("authority :", settings.azure_authority)
print("redirect  :", settings.azure_redirect_uri)
print("scopes    :")
for s in settings.graph_scopes:
    print("   -", s)

## Web flow (auth code + PKCE)

1. `begin_web_auth()` returns a flow dict containing `auth_uri` (send the browser
   there) plus the PKCE `code_verifier` and CSRF `state`.
2. Microsoft redirects back to `/auth/callback`.
3. `complete_web_auth(flow, params)` exchanges the code for a token and returns a
   `TenantSession`.

## CLI flow

`cli_interactive_auth()` opens a browser via a loopback redirect, falling back to
device-code for headless containers.

Both flows converge on a `TenantSession`, whose `__repr__` redacts the token:

In [ ]:
from m365_review.core.auth import TenantSession

demo = TenantSession(access_token="SUPER-SECRET-DO-NOT-LOG", tenant_id="contoso-tid", username="admin@contoso.com")
print(repr(demo))   # note: access_token is <redacted>